# Phase D-2: BoTorch Inverse Design

## FNO Surrogate → BoTorch qNEHVI → Optimal Design

```
목표 성능 (MTF > 0.6, skew < 0.1)
    ↓
BoTorch qNEHVI (multi-objective)
    ↓ 설계변수 후보 제안
FNO Surrogate (1ms/query)
    ↓ PSF 예측
Metrics (MTF, skew, T)
    ↓ GP 모델 업데이트
    ↓ 반복 (30 iterations)
Pareto Front → Top-5 → 최적 설계
```

### 3개 목적함수
| # | 목적 | 방향 | 의미 |
|---|------|------|------|
| 1 | MTF@ridge | 최대화 | 해상도 |
| 2 | Throughput | 최대화 | 밝기 |
| 3 | -|Skewness| | 최대화(=|skew| 최소화) | 대칭성 |

In [ ]:
import sys, json, time
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
print('Ready')

---
## Run Inverse Design

In [ ]:
from backend.core.botorch_optimizer import run_inverse_design

fno_ckpt = PROJECT_ROOT / 'checkpoints' / 'fno_surrogate.pt'

if not fno_ckpt.exists():
    print('FNO not trained yet. Run:')
    print('  python scripts/distill_fno.py --epochs 500')
else:
    result = run_inverse_design(
        fno_checkpoint=str(fno_ckpt),
        n_initial=30,
        n_iterations=30,
        batch_size=4,
        theta_deg=0.0,
    )

---
## Results Visualization

In [ ]:
if fno_ckpt.exists():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    obj = result.all_objectives
    pareto = result.pareto_objectives
    
    # MTF vs Throughput
    ax = axes[0]
    ax.scatter(obj[:, 0], obj[:, 1], c='gray', s=20, alpha=0.5, label='All')
    if len(pareto) > 0:
        ax.scatter(pareto[:, 0], pareto[:, 1], c='red', s=50, zorder=5, label='Pareto')
    ax.set_xlabel('MTF@ridge')
    ax.set_ylabel('Throughput')
    ax.set_title('MTF vs Throughput')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # MTF vs Skewness
    ax = axes[1]
    ax.scatter(obj[:, 0], -obj[:, 2], c='gray', s=20, alpha=0.5, label='All')
    if len(pareto) > 0:
        ax.scatter(pareto[:, 0], -pareto[:, 2], c='red', s=50, zorder=5, label='Pareto')
    ax.set_xlabel('MTF@ridge')
    ax.set_ylabel('|Skewness|')
    ax.set_title('MTF vs |Skewness|')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Design space explored
    ax = axes[2]
    params = result.all_params
    ax.scatter(params[:, 2], params[:, 3], c=obj[:, 0], cmap='RdYlGn', s=20, alpha=0.7)
    if len(result.pareto_params) > 0:
        ax.scatter(result.pareto_params[:, 2], result.pareto_params[:, 3],
                   c='red', s=80, marker='*', zorder=5, label='Pareto')
    ax.set_xlabel('w1 (um)')
    ax.set_ylabel('w2 (um)')
    ax.set_title('Design Space (color=MTF)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'BoTorch Inverse Design ({result.elapsed_sec:.0f}s, '
                 f'{len(result.all_params)} evals, {len(result.pareto_params)} Pareto)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Best design
    print('Best design (by MTF):')
    for k, v in result.best_metrics.items():
        print(f'  {k}: {v:.4f}')

---
## Summary

### Phase D Pipeline
```
PINN (Phase C)  →  FNO Distillation  →  BoTorch qNEHVI  →  Optimal Design
  ~4-8h GPU          ~10min               ~5min              즉시
  (한번만)            (한번만)             (반복 가능)         (결과)
```

### Next: Phase E (Design Studio UI + FastAPI)